# 04 — Coverage of the measured matrix

> **Discovery stage — hypothesis-generating, not confirmatory.** The Discovery-9 panel (9 targets × 30 measured ligands, **no decoys**) is used to *pick* the GBSA scoring combo and to *generate* the hypothesis that MM-GBSA gives an early-enrichment edge over docking. Significance here is subject to combo selection (winner's curse) and is reported as a **trend**. The confirmatory claim is deferred to the pre-registered locked **n=18** validation (`VALIDATION_PLAN.md`).


> **Reader guide — where this notebook sits in the study.**
>
> **Part of:** *Orientation section* — no experiment.
>
> **Question this NB answers:** *for each target, how complete is the 30-ligand column
> (docking done? MD done? GBSA rescored?), and where are the gaps?*
>
> **Method:** availability matrix per (target, ligand, stage).
>
> **Reproducibility contract:** only reads reference CSVs; no numeric analysis, only counts.
>
> **Environment:** requires `pip install -e .` from the repo root.

In [ ]:
NB_STEM = "03_coverage"
# ===== repo-relative setup — reruns from a fresh clone, no absolute paths =====
import sys, json
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib.axes import Axes
from matplotlib.figure import Figure
from scipy import stats
from IPython.display import display

# make the in-repo package importable even without `pip install -e` (fresh clone)
_here = Path.cwd()
_root = next((p for p in [_here, *_here.parents] if (p / "pyproject.toml").is_file()), _here)
if str(_root / "src") not in sys.path:
    sys.path.insert(0, str(_root / "src"))

from gbsabench.paths import RAW, DERIVED, FIGURES, TABLES
from gbsabench import style, metrics
from gbsabench.io import load
style.apply_style()
NAVY, GOLD, GREY, GREYD, CREAM, WHITE = style.NAVY, style.GOLD, style.GREY, style.GREY_DASH, style.CREAM, style.WHITE

# Publication style: in-figure titles are suppressed. The premise -- "markdown + captions
# carry the description" -- was FALSE: no caption file existed, so 18 set_title calls were
# silently deleted across the package, including five panel labels in the main deliverable
# figure and the word PARTIAL on the partial-coverage map (referee finding, rounds 6-7).
#
# Titles are still suppressed for publication, but they are now RECORDED rather than
# discarded, and figures/CAPTIONS.md is generated from what was captured -- so the
# description really does exist somewhere a reader can reach.
_SUPPRESSED_TITLES = []
def _capture_title(self, *a, **k):
    if a and isinstance(a[0], str) and a[0].strip():
        _SUPPRESSED_TITLES.append((NB_STEM, a[0].strip()))
def _capture_suptitle(self, *a, **k):
    if a and isinstance(a[0], str) and a[0].strip():
        _SUPPRESSED_TITLES.append((NB_STEM, a[0].strip()))
Axes.set_title  = _capture_title
Figure.suptitle = _capture_suptitle

# capture every figure as it is created, so the last cell can export them all to figures/
_CREATED_FIGS = []
if not getattr(plt.subplots, "_gbsa_wrapped", False):   # idempotent: never re-wrap on a dirty kernel
    _orig_subplots, _orig_figure = plt.subplots, plt.figure
    def _register(fig):
        # plt.subplots() calls plt.figure() internally, so a figure made with subplots
        # hit BOTH wrappers and was captured twice -- which is why every notebook
        # exported byte-identical fig1/fig2 pairs (referee finding, iteration 1).
        if not any(fig is seen for seen in _CREATED_FIGS):
            _CREATED_FIGS.append(fig)
        return fig
    def _cap_subplots(*a, **k):
        result = _orig_subplots(*a, **k); _register(result[0]); return result
    def _cap_figure(*a, **k):
        return _register(_orig_figure(*a, **k))
    _cap_subplots._gbsa_wrapped = _cap_figure._gbsa_wrapped = True
    plt.subplots, plt.figure = _cap_subplots, _cap_figure

# reviewer-friendly TABLE HEADERS (display only; the raw short column names stay unchanged underneath)
FRIENDLY = {
    "target": "target", "n": "ligands", "actives": "actives", "inactives": "inactives",
    "total": "ligands (total)", "gbsa_measured": "measured", "gbsa_pct": "measured %",
    "gbsa_actives": "actives", "gbsa_inactives": "inactives",
    "tau_gbsa": "Kendall τ (GBSA)", "tau_dock": "Kendall τ (dock)",
    "bedroc_gbsa": "BEDROC (GBSA)", "bedroc_dock": "BEDROC (dock)",
    "auc_gbsa": "ROC-AUC (GBSA)", "auc_dock": "ROC-AUC (dock)", "delong_p": "DeLong p",
    "metric": "metric", "gbsa_median": "GBSA (median)", "dock_median": "dock (median)",
    "wins": "GBSA wins", "p_onesided": "one-sided p", "p_twosided": "two-sided p",
    "quantity": "quantity", "value": "value", "role": "role", "description": "description",
    "factor": "factor", "eta2_tau": "η² on τ", "eta2_bedroc": "η² on BEDROC",
    "eta2_bedroc_target_blocked": "η² on BEDROC (target-blocked)",
    "eta2_nsday_MECHANICAL": "η² on ns/day (mechanical)", "eta2_steps_per_sec_REAL": "η² on steps/sec (real)",
    "dt_fs": "dt (fs)", "attempts": "attempts", "ok": "succeeded", "fail_rate_pct": "fail-rate %",
    "median_nsday_OK": "median ns/day (OK runs)", "usable_nsday_after_failures": "usable ns/GPU-day",
    "estimator": "estimator", "p_with_4L7G": "p (with 4L7G)", "p_without_4L7G": "p (without 4L7G)",
    "pdb": "PDB", "family": "family", "crystal_ligand": "crystal ligand",
    "crystal_ligand_rscc": "RSCC", "resolution_A": "resolution (Å)", "split": "split",
}
if not getattr(pd.DataFrame._repr_html_, "_gbsa_wrapped", False):   # idempotent: never re-wrap on a dirty kernel
    _orig_html, _orig_repr = pd.DataFrame._repr_html_, pd.DataFrame.__repr__
    def _friendly_html(self): return _orig_html(self.rename(columns={c: FRIENDLY.get(c, c) for c in self.columns}))
    def _friendly_repr(self): return _orig_repr(self.rename(columns={c: FRIENDLY.get(c, c) for c in self.columns}))
    _friendly_html._gbsa_wrapped = _friendly_repr._gbsa_wrapped = True
    pd.DataFrame._repr_html_, pd.DataFrame.__repr__ = _friendly_html, _friendly_repr

SELECTED_COMBO = "igb2_di4_salt0.15_st0.0072"   # the locked best-of-48 physics combo (see notebook 03)

def _figs():
    """Export every figure this notebook created to figures/ (the last-cell convention)."""
    for i, fig in enumerate(_CREATED_FIGS, start=1):
        fig.savefig(FIGURES / f"{NB_STEM}_fig{i}.png")
    print("exported", len(_CREATED_FIGS), "figure(s) to figures/")


## Coverage

**What we do.** Report exactly how much of the 9 × 30 GBSA matrix was actually measured before the cluster went into maintenance. Split by target and by active/inactive label, so every later notebook is honest about the N it stands on.

**How.** A complex counts as *measured* if its selected-combo ΔG is in the raw GBSA table. We count measured complexes per target against the 30 designed ligands, and within each target split by `is_active` (top-10 strongest pKi = active, 20 weaker = inactive; all are measured binders — no decoys).

**One step:** raw rows → a coverage table. No plot; the table is the artefact.


**Step 1 — raw data → table.** Measured complexes per target, plus the active/inactive split, recomputed from `data/raw/`.


In [ ]:
gbsa = load("gbsa_dG_raw"); meta = load("metadata")
measured_ids = set(gbsa[gbsa.combo == SELECTED_COMBO].complex_id.unique())

coverage_rows = []
for target, target_ligands in meta.groupby("target"):
    measured = target_ligands[target_ligands.complex_id.isin(measured_ids)]
    coverage_rows.append({"target": target,
                          "total": len(target_ligands),
                          "gbsa_measured": len(measured),
                          "gbsa_pct": round(100 * len(measured) / len(target_ligands)),
                          "gbsa_actives": int(measured.is_active.sum()),
                          "gbsa_inactives": int((~measured.is_active).sum())})
coverage_table = pd.DataFrame(coverage_rows).sort_values("target").reset_index(drop=True)
print(f"overall: {len(measured_ids)} of {len(meta)} complexes measured "
      f"({100*len(measured_ids)/len(meta):.0f}%); {(coverage_table.gbsa_measured>0).sum()} of 9 targets covered")

# Persist the 9x30 coverage table this notebook already builds. Until round 6 this table was READ by verify.py and by
# later notebooks but WRITTEN by nothing: deleting data/derived/ and re-running
# the reading order did not bring it back. Found by our own lineage audit.
coverage_table.to_csv(DERIVED / "coverage.csv", index=False)
coverage_table

**Interpretation.** Eight of the nine targets are covered. **4A5S is at 0 % coverage and is excluded** from every panel statistic, so all panel tests run at **n = 8 targets**. Per-target N ranges 21–28 (as few as 6 measured actives on 9SI4), so the lowest-coverage targets carry the most sampling noise. Partial-coverage snapshot from cluster maintenance — reported as-is, not back-filled.


## Export figures
Save this notebook's figures to `figures/`.


In [ ]:
_figs()


In [ ]:
FIGURE_CAPTIONS = {
}

# ---- captions, keyed by FILENAME ----------------------------------------------------
# The premise printed at the top of every notebook is that suppressed in-figure titles are
# carried by figures/CAPTIONS.md instead. That premise has been false twice: first no caption
# file existed at all, then titles were captured into a list nothing read. The third failure
# was subtler and is fixed here -- the file recorded TITLES but not FILENAMES, so a reader
# holding a PNG could not find its caption, and most notebooks contributed nothing because
# their titles had already been deleted rather than suppressed. Every figure this notebook
# writes now gets a line naming the file; captured titles are appended where they exist.
# verify.py section 16 asserts the coverage, and it is the first check in this package that
# can fail because of a picture (referee, six rounds).
_cap = FIGURES / "CAPTIONS.md"
_mine = sorted(p for p in FIGURES.rglob("*.png") if p.name.startswith(NB_STEM + "_"))
_prev = _cap.read_text() if _cap.exists() else ""
_keep = [l for l in _prev.splitlines()
         if l.startswith("- ") and f"**{NB_STEM}**" not in l]
_lines = []
for _p in _mine:
    _rel = _p.relative_to(FIGURES).as_posix()
    # match on the full basename first, then on the suffix after NB_STEM, because
    # notebooks 02-07 export as {stem}_fig{n} while 08-10 name each figure.
    _d = FIGURE_CAPTIONS.get(_p.name) or FIGURE_CAPTIONS.get(
        _p.stem.removeprefix(NB_STEM + "_"), "")
    _lines.append(f"- `{_rel}` — **{NB_STEM}** — {_d}" if _d
                  else f"- `{_rel}` — **{NB_STEM}** — NO CAPTION WRITTEN")

_lines += [f"- **{NB_STEM}** — suppressed title: {t}" for _, t in _SUPPRESSED_TITLES]
_cap.write_text("# Figure captions\n\nOne line per shipped figure, naming the file, plus any\n"
                "in-figure title suppressed for publication.\n\n"
                + "\n".join(sorted(set(_keep + _lines))) + "\n")
print(f"captions: {len(_mine)} figure(s) and {len(_SUPPRESSED_TITLES)} suppressed title(s) "
      f"recorded in {_cap.name}")
